# Reto SQL: seguimiento histórico de campeones y posiciones en un archivo olímpico

## Contexto del problema

Una organización deportiva guarda un archivo sintético de medallas de cuatro ediciones de unos Juegos de Verano. Cada fila representa una medalla concedida.

Necesitas preparar un informe único que permita revisar dos cosas a la vez: cómo se posicionan las medallas de oro dentro del archivo y cómo cambia el campeón de un evento a lo largo de los años. El informe también debe permitir revisar posiciones dentro de cada combinación de año y país para las medallas de oro femeninas.

El reto debe resolverse con una consulta SQL que trabaje sobre el mismo conjunto de datos y produzca las salidas pedidas, sin eliminar las filas originales mediante una agregación.

## Datos disponibles

La tabla se llama `summer_medals` y contiene estas columnas:

| Columna | Descripción |
|---|---|
| `year` | Año de la edición |
| `city` | Ciudad de la edición |
| `sport` | Deporte |
| `discipline` | Disciplina |
| `event` | Evento |
| `athlete` | Identificador sintético del atleta |
| `country` | Código del país |
| `gender` | Género (`Men` o `Women`) |
| `medal` | Tipo de medalla (`Gold`, `Silver`, `Bronze`) |

Los datos son sintéticos y se crean dentro del notebook. No necesitas ningún archivo externo.

La documentación entregada trabaja PostgreSQL. Para que este notebook sea autocontenido y no necesite instalar ni conectar ningún servicio, la ejecución usa el módulo `sqlite3` incluido en Python. Las expresiones SQL evaluadas pertenecen al subconjunto común que mantiene la misma semántica para este reto: `OVER`, `ROW_NUMBER`, `LAG`, `ORDER BY`, `PARTITION BY` y CTEs.

## Objetivo

Construye un informe SQL único que responda a las tres necesidades siguientes. Debes decidir cómo organizar las consultas para que cada resultado sea correcto y pueda auditarse con sus datos originales.

In [13]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE summer_medals (
    year INTEGER,
    city TEXT,
    sport TEXT,
    discipline TEXT,
    event TEXT,
    athlete TEXT,
    country TEXT,
    gender TEXT,
    medal TEXT
)
""")


In [14]:
rows = [(2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'M_2008_Discus_Throw_Gold_01', 'GER', 'Men', 'Gold'), (2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'M_2008_Discus_Throw_Silver_02', 'FRA', 'Men', 'Silver'), (2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'M_2008_Discus_Throw_Bronze_03', 'USA', 'Men', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'M_2008_Triple_Jump_Gold_01', 'POR', 'Men', 'Gold'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'M_2008_Triple_Jump_Silver_02', 'USA', 'Men', 'Silver'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'M_2008_Triple_Jump_Bronze_03', 'CHN', 'Men', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'M_2008_100M_Gold_01', 'USA', 'Men', 'Gold'), (2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'M_2008_100M_Silver_02', 'ESP', 'Men', 'Silver'), (2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'M_2008_100M_Bronze_03', 'CAN', 'Men', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Sprints', '200M', 'M_2008_200M_Gold_01', 'USA', 'Men', 'Gold'), (2008, 'Asteria', 'Athletics', 'Sprints', '200M', 'M_2008_200M_Silver_02', 'JPN', 'Men', 'Silver'), (2008, 'Asteria', 'Athletics', 'Sprints', '200M', 'M_2008_200M_Bronze_03', 'BRA', 'Men', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Long Jump', 'M_2008_Long_Jump_Gold_01', 'CUB', 'Men', 'Gold'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Long Jump', 'M_2008_Long_Jump_Silver_02', 'CAN', 'Men', 'Silver'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Long Jump', 'M_2008_Long_Jump_Bronze_03', 'FRA', 'Men', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Throws', 'Discus Throw', 'M_2012_Discus_Throw_Gold_01', 'GER', 'Men', 'Gold'), (2012, 'Borealis', 'Athletics', 'Throws', 'Discus Throw', 'M_2012_Discus_Throw_Silver_02', 'USA', 'Men', 'Silver'), (2012, 'Borealis', 'Athletics', 'Throws', 'Discus Throw', 'M_2012_Discus_Throw_Bronze_03', 'JPN', 'Men', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Triple Jump', 'M_2012_Triple_Jump_Gold_01', 'POR', 'Men', 'Gold'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Triple Jump', 'M_2012_Triple_Jump_Silver_02', 'CHN', 'Men', 'Silver'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Triple Jump', 'M_2012_Triple_Jump_Bronze_03', 'CUB', 'Men', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Sprints', '100M', 'M_2012_100M_Gold_01', 'USA', 'Men', 'Gold'), (2012, 'Borealis', 'Athletics', 'Sprints', '100M', 'M_2012_100M_Silver_02', 'CAN', 'Men', 'Silver'), (2012, 'Borealis', 'Athletics', 'Sprints', '100M', 'M_2012_100M_Bronze_03', 'SWE', 'Men', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Sprints', '200M', 'M_2012_200M_Gold_01', 'USA', 'Men', 'Gold'), (2012, 'Borealis', 'Athletics', 'Sprints', '200M', 'M_2012_200M_Silver_02', 'BRA', 'Men', 'Silver'), (2012, 'Borealis', 'Athletics', 'Sprints', '200M', 'M_2012_200M_Bronze_03', 'ITA', 'Men', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Long Jump', 'M_2012_Long_Jump_Gold_01', 'CUB', 'Men', 'Gold'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Long Jump', 'M_2012_Long_Jump_Silver_02', 'FRA', 'Men', 'Silver'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Long Jump', 'M_2012_Long_Jump_Bronze_03', 'USA', 'Men', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Throws', 'Discus Throw', 'M_2016_Discus_Throw_Gold_01', 'ESP', 'Men', 'Gold'), (2016, 'Caspia', 'Athletics', 'Throws', 'Discus Throw', 'M_2016_Discus_Throw_Silver_02', 'JPN', 'Men', 'Silver'), (2016, 'Caspia', 'Athletics', 'Throws', 'Discus Throw', 'M_2016_Discus_Throw_Bronze_03', 'BRA', 'Men', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Triple Jump', 'M_2016_Triple_Jump_Gold_01', 'ITA', 'Men', 'Gold'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Triple Jump', 'M_2016_Triple_Jump_Silver_02', 'CUB', 'Men', 'Silver'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Triple Jump', 'M_2016_Triple_Jump_Bronze_03', 'FRA', 'Men', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Sprints', '100M', 'M_2016_100M_Gold_01', 'JAM', 'Men', 'Gold'), (2016, 'Caspia', 'Athletics', 'Sprints', '100M', 'M_2016_100M_Silver_02', 'BRA', 'Men', 'Silver'), (2016, 'Caspia', 'Athletics', 'Sprints', '100M', 'M_2016_100M_Bronze_03', 'ITA', 'Men', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Sprints', '200M', 'M_2016_200M_Gold_01', 'USA', 'Men', 'Gold'), (2016, 'Caspia', 'Athletics', 'Sprints', '200M', 'M_2016_200M_Silver_02', 'ITA', 'Men', 'Silver'), (2016, 'Caspia', 'Athletics', 'Sprints', '200M', 'M_2016_200M_Bronze_03', 'ESP', 'Men', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Long Jump', 'M_2016_Long_Jump_Gold_01', 'JPN', 'Men', 'Gold'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Long Jump', 'M_2016_Long_Jump_Silver_02', 'USA', 'Men', 'Silver'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Long Jump', 'M_2016_Long_Jump_Bronze_03', 'CHN', 'Men', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Throws', 'Discus Throw', 'M_2020_Discus_Throw_Gold_01', 'ESP', 'Men', 'Gold'), (2020, 'Draconis', 'Athletics', 'Throws', 'Discus Throw', 'M_2020_Discus_Throw_Silver_02', 'BRA', 'Men', 'Silver'), (2020, 'Draconis', 'Athletics', 'Throws', 'Discus Throw', 'M_2020_Discus_Throw_Bronze_03', 'ITA', 'Men', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Triple Jump', 'M_2020_Triple_Jump_Gold_01', 'ITA', 'Men', 'Gold'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Triple Jump', 'M_2020_Triple_Jump_Silver_02', 'FRA', 'Men', 'Silver'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Triple Jump', 'M_2020_Triple_Jump_Bronze_03', 'GER', 'Men', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Sprints', '100M', 'M_2020_100M_Gold_01', 'USA', 'Men', 'Gold'), (2020, 'Draconis', 'Athletics', 'Sprints', '100M', 'M_2020_100M_Silver_02', 'POR', 'Men', 'Silver'), (2020, 'Draconis', 'Athletics', 'Sprints', '100M', 'M_2020_100M_Bronze_03', 'CHN', 'Men', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Sprints', '200M', 'M_2020_200M_Gold_01', 'CAN', 'Men', 'Gold'), (2020, 'Draconis', 'Athletics', 'Sprints', '200M', 'M_2020_200M_Silver_02', 'GER', 'Men', 'Silver'), (2020, 'Draconis', 'Athletics', 'Sprints', '200M', 'M_2020_200M_Bronze_03', 'JPN', 'Men', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Long Jump', 'M_2020_Long_Jump_Gold_01', 'CHN', 'Men', 'Gold'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Long Jump', 'M_2020_Long_Jump_Silver_02', 'JPN', 'Men', 'Silver'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Long Jump', 'M_2020_Long_Jump_Bronze_03', 'BRA', 'Men', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'W_2008_Discus_Throw_Gold_01', 'FRA', 'Women', 'Gold'), (2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'W_2008_Discus_Throw_Silver_02', 'POR', 'Women', 'Silver'), (2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'W_2008_Discus_Throw_Bronze_03', 'ESP', 'Women', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'W_2008_Triple_Jump_Gold_01', 'ESP', 'Women', 'Gold'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'W_2008_Triple_Jump_Silver_02', 'POR', 'Women', 'Silver'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'W_2008_Triple_Jump_Bronze_03', 'CHN', 'Women', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'W_2008_100M_Gold_01', 'CHN', 'Women', 'Gold'), (2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'W_2008_100M_Silver_02', 'USA', 'Women', 'Silver'), (2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'W_2008_100M_Bronze_03', 'JPN', 'Women', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Sprints', '200M', 'W_2008_200M_Gold_01', 'JPN', 'Women', 'Gold'), (2008, 'Asteria', 'Athletics', 'Sprints', '200M', 'W_2008_200M_Silver_02', 'GER', 'Women', 'Silver'), (2008, 'Asteria', 'Athletics', 'Sprints', '200M', 'W_2008_200M_Bronze_03', 'CAN', 'Women', 'Bronze'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Long Jump', 'W_2008_Long_Jump_Gold_01', 'CHN', 'Women', 'Gold'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Long Jump', 'W_2008_Long_Jump_Silver_02', 'ESP', 'Women', 'Silver'), (2008, 'Asteria', 'Athletics', 'Jumps', 'Long Jump', 'W_2008_Long_Jump_Bronze_03', 'CUB', 'Women', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Throws', 'Discus Throw', 'W_2012_Discus_Throw_Gold_01', 'ESP', 'Women', 'Gold'), (2012, 'Borealis', 'Athletics', 'Throws', 'Discus Throw', 'W_2012_Discus_Throw_Silver_02', 'JPN', 'Women', 'Silver'), (2012, 'Borealis', 'Athletics', 'Throws', 'Discus Throw', 'W_2012_Discus_Throw_Bronze_03', 'BRA', 'Women', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Triple Jump', 'W_2012_Triple_Jump_Gold_01', 'CHN', 'Women', 'Gold'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Triple Jump', 'W_2012_Triple_Jump_Silver_02', 'CAN', 'Women', 'Silver'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Triple Jump', 'W_2012_Triple_Jump_Bronze_03', 'SWE', 'Women', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Sprints', '100M', 'W_2012_100M_Gold_01', 'CHN', 'Women', 'Gold'), (2012, 'Borealis', 'Athletics', 'Sprints', '100M', 'W_2012_100M_Silver_02', 'CUB', 'Women', 'Silver'), (2012, 'Borealis', 'Athletics', 'Sprints', '100M', 'W_2012_100M_Bronze_03', 'FRA', 'Women', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Sprints', '200M', 'W_2012_200M_Gold_01', 'JPN', 'Women', 'Gold'), (2012, 'Borealis', 'Athletics', 'Sprints', '200M', 'W_2012_200M_Silver_02', 'BRA', 'Women', 'Silver'), (2012, 'Borealis', 'Athletics', 'Sprints', '200M', 'W_2012_200M_Bronze_03', 'ITA', 'Women', 'Bronze'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Long Jump', 'W_2012_Long_Jump_Gold_01', 'CHN', 'Women', 'Gold'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Long Jump', 'W_2012_Long_Jump_Silver_02', 'SWE', 'Women', 'Silver'), (2012, 'Borealis', 'Athletics', 'Jumps', 'Long Jump', 'W_2012_Long_Jump_Bronze_03', 'POR', 'Women', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Throws', 'Discus Throw', 'W_2016_Discus_Throw_Gold_01', 'ESP', 'Women', 'Gold'), (2016, 'Caspia', 'Athletics', 'Throws', 'Discus Throw', 'W_2016_Discus_Throw_Silver_02', 'FRA', 'Women', 'Silver'), (2016, 'Caspia', 'Athletics', 'Throws', 'Discus Throw', 'W_2016_Discus_Throw_Bronze_03', 'USA', 'Women', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Triple Jump', 'W_2016_Triple_Jump_Gold_01', 'ESP', 'Women', 'Gold'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Triple Jump', 'W_2016_Triple_Jump_Silver_02', 'ITA', 'Women', 'Silver'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Triple Jump', 'W_2016_Triple_Jump_Bronze_03', 'GER', 'Women', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Sprints', '100M', 'W_2016_100M_Gold_01', 'JPN', 'Women', 'Gold'), (2016, 'Caspia', 'Athletics', 'Sprints', '100M', 'W_2016_100M_Silver_02', 'POR', 'Women', 'Silver'), (2016, 'Caspia', 'Athletics', 'Sprints', '100M', 'W_2016_100M_Bronze_03', 'ESP', 'Women', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Sprints', '200M', 'W_2016_200M_Gold_01', 'JPN', 'Women', 'Gold'), (2016, 'Caspia', 'Athletics', 'Sprints', '200M', 'W_2016_200M_Silver_02', 'USA', 'Women', 'Silver'), (2016, 'Caspia', 'Athletics', 'Sprints', '200M', 'W_2016_200M_Bronze_03', 'CHN', 'Women', 'Bronze'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Long Jump', 'W_2016_Long_Jump_Gold_01', 'CHN', 'Women', 'Gold'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Long Jump', 'W_2016_Long_Jump_Silver_02', 'GER', 'Women', 'Silver'), (2016, 'Caspia', 'Athletics', 'Jumps', 'Long Jump', 'W_2016_Long_Jump_Bronze_03', 'CAN', 'Women', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Throws', 'Discus Throw', 'W_2020_Discus_Throw_Gold_01', 'ESP', 'Women', 'Gold'), (2020, 'Draconis', 'Athletics', 'Throws', 'Discus Throw', 'W_2020_Discus_Throw_Silver_02', 'CHN', 'Women', 'Silver'), (2020, 'Draconis', 'Athletics', 'Throws', 'Discus Throw', 'W_2020_Discus_Throw_Bronze_03', 'CUB', 'Women', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Triple Jump', 'W_2020_Triple_Jump_Gold_01', 'CHN', 'Women', 'Gold'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Triple Jump', 'W_2020_Triple_Jump_Silver_02', 'JPN', 'Women', 'Silver'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Triple Jump', 'W_2020_Triple_Jump_Bronze_03', 'BRA', 'Women', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Sprints', '100M', 'W_2020_100M_Gold_01', 'CHN', 'Women', 'Gold'), (2020, 'Draconis', 'Athletics', 'Sprints', '100M', 'W_2020_100M_Silver_02', 'CAN', 'Women', 'Silver'), (2020, 'Draconis', 'Athletics', 'Sprints', '100M', 'W_2020_100M_Bronze_03', 'SWE', 'Women', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Sprints', '200M', 'W_2020_200M_Gold_01', 'JPN', 'Women', 'Gold'), (2020, 'Draconis', 'Athletics', 'Sprints', '200M', 'W_2020_200M_Silver_02', 'CUB', 'Women', 'Silver'), (2020, 'Draconis', 'Athletics', 'Sprints', '200M', 'W_2020_200M_Bronze_03', 'FRA', 'Women', 'Bronze'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Long Jump', 'W_2020_Long_Jump_Gold_01', 'CHN', 'Women', 'Gold'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Long Jump', 'W_2020_Long_Jump_Silver_02', 'BRA', 'Women', 'Silver'), (2020, 'Draconis', 'Athletics', 'Jumps', 'Long Jump', 'W_2020_Long_Jump_Bronze_03', 'ITA', 'Women', 'Bronze')]
conn.executemany("""
INSERT INTO summer_medals
    (year, city, sport, discipline, event, athlete, country, gender, medal)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""", rows)
conn.commit()
print(f"Filas cargadas: {conn.execute('SELECT COUNT(*) FROM summer_medals').fetchone()[0]}")


Filas cargadas: 120


In [15]:
# Explora libremente los datos antes de resolver el reto.

cur = conn.execute("SELECT * FROM summer_medals LIMIT 12")
print([d[0] for d in cur.description])
for row in cur.fetchall():
    print(row)


['year', 'city', 'sport', 'discipline', 'event', 'athlete', 'country', 'gender', 'medal']
(2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'M_2008_Discus_Throw_Gold_01', 'GER', 'Men', 'Gold')
(2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'M_2008_Discus_Throw_Silver_02', 'FRA', 'Men', 'Silver')
(2008, 'Asteria', 'Athletics', 'Throws', 'Discus Throw', 'M_2008_Discus_Throw_Bronze_03', 'USA', 'Men', 'Bronze')
(2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'M_2008_Triple_Jump_Gold_01', 'POR', 'Men', 'Gold')
(2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'M_2008_Triple_Jump_Silver_02', 'USA', 'Men', 'Silver')
(2008, 'Asteria', 'Athletics', 'Jumps', 'Triple Jump', 'M_2008_Triple_Jump_Bronze_03', 'CHN', 'Men', 'Bronze')
(2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'M_2008_100M_Gold_01', 'USA', 'Men', 'Gold')
(2008, 'Asteria', 'Athletics', 'Sprints', '100M', 'M_2008_100M_Silver_02', 'ESP', 'Men', 'Silver')
(2008, 'Asteria', 'Athletics', 'Sprints', '100M',

## Requisitos del reto

### 1. Posición global de las medallas de oro

Partiendo solo de las medallas de oro, añade una posición que siga este criterio: años del más reciente al más antiguo y, dentro de cada año, eventos en orden alfabético.

Después, presenta ese mismo resultado ordenado por país y, dentro de cada país, por la posición calculada. La posición debe conservar el criterio anterior aunque cambie el orden visual de las filas.

### 2. Historial de campeones por evento

Crea una salida para las medallas de oro masculinas de `Discus Throw` y `Triple Jump` que muestre, por cada año y evento, el campeón actual y el campeón del año anterior del mismo evento.

En el primer año de cada evento no debe aparecer un campeón anterior. El resultado debe permitir identificar claramente los cambios de campeón y las repeticiones de un mismo campeón en años consecutivos.

### 3. Posición de las medallas de oro femeninas dentro de cada año y país

Para cada medalla de oro femenina, añade una posición dentro de su combinación de año y país. Esa posición debe empezar en 1 para cada nueva combinación y seguir un orden alfabético por evento.

Presenta la salida ordenada por año, país y posición.

### Criterios de aceptación

El informe debe mantener las filas de cada consulta y añadir las columnas de análisis necesarias. Las posiciones y comparaciones deben respetar exactamente sus criterios de orden y agrupación lógica. Los inicios de cada historial independiente deben quedar separados y no deben arrastrar información de otro evento, año o país.

## Espacio para la resolución del usuario

Escribe y ejecuta aquí tu propia solución. Puedes dividirla en varias celdas de código, revisar las salidas, cambiar el enfoque y volver a ejecutar. El espacio es deliberadamente libre para que tomes tus propias decisiones.

In [19]:
import pandas as pd

q1 = '''
    SELECT 
        country, year, event,
        ROW_NUMBER() OVER(
            ORDER BY year ASC, event ASC
        ) AS posicion
    FROM summer_medals
    WHERE medal = 'Gold'
    ORDER BY country, posicion

;
'''

print(pd.read_sql_query(q1, conn))

   country  year         event  posicion
0      CAN  2020          200M        33
1      CHN  2008          100M         2
2      CHN  2008     Long Jump         8
3      CHN  2012          100M        12
4      CHN  2012     Long Jump        18
5      CHN  2012   Triple Jump        20
6      CHN  2016     Long Jump        28
7      CHN  2020          100M        32
8      CHN  2020     Long Jump        37
9      CHN  2020     Long Jump        38
10     CHN  2020   Triple Jump        40
11     CUB  2008     Long Jump         7
12     CUB  2012     Long Jump        17
13     ESP  2008   Triple Jump        10
14     ESP  2012  Discus Throw        16
15     ESP  2016  Discus Throw        25
16     ESP  2016  Discus Throw        26
17     ESP  2016   Triple Jump        30
18     ESP  2020  Discus Throw        35
19     ESP  2020  Discus Throw        36
20     FRA  2008  Discus Throw         6
21     GER  2008  Discus Throw         5
22     GER  2012  Discus Throw        15
23     ITA  2016

In [ ]:
# Puedes usar esta celda para comprobaciones adicionales de tu solución.


# Solución - consultar después de intentar

La solución siguiente usa una consulta base para cada entregable y aplica las funciones de ventana sobre las filas que deben conservarse. Las consultas están separadas solo para hacer auditable el mismo reto; no representan ejercicios independientes.

In [ ]:
import sqlite3
from pprint import pprint

# La conexión y la tabla ya han sido creadas antes de llegar a la solución.
# Esta función muestra las filas de cualquier consulta de forma legible.
def run_sql(sql):
    cur = conn.execute(sql)
    columns = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(" | ".join(columns))
    print("-" * (len(" | ".join(columns)) + 4))
    for row in rows:
        print(" | ".join("NULL" if v is None else str(v) for v in row))
    print(f"Filas: {len(rows)}")
    return columns, rows


### Solución del requisito 1

La posición se calcula con un orden dentro de la ventana. Después, otro orden independiente organiza la salida final. Esto hace visible la diferencia entre el orden que determina la posición y el orden que determina la presentación.

In [ ]:
sql_q1 = """
SELECT
    year,
    event,
    country,
    ROW_NUMBER() OVER (
        ORDER BY year DESC, event ASC
    ) AS position_global
FROM summer_medals
WHERE medal = 'Gold'
ORDER BY country ASC, position_global ASC;
"""
q1_columns, q1_rows = run_sql(sql_q1)


year | event | country | position_global
--------------------------------------------
2020 | 200M | CAN | 3
2020 | 100M | CHN | 2
2020 | Long Jump | CHN | 7
2020 | Long Jump | CHN | 8
2020 | Triple Jump | CHN | 10
2016 | Long Jump | CHN | 18
2012 | 100M | CHN | 22
2012 | Long Jump | CHN | 28
2012 | Triple Jump | CHN | 30
2008 | 100M | CHN | 32
2008 | Long Jump | CHN | 38
2012 | Long Jump | CUB | 27
2008 | Long Jump | CUB | 37
2020 | Discus Throw | ESP | 5
2020 | Discus Throw | ESP | 6
2016 | Discus Throw | ESP | 15
2016 | Discus Throw | ESP | 16
2016 | Triple Jump | ESP | 20
2012 | Discus Throw | ESP | 26
2008 | Triple Jump | ESP | 40
2008 | Discus Throw | FRA | 36
2012 | Discus Throw | GER | 25
2008 | Discus Throw | GER | 35
2020 | Triple Jump | ITA | 9
2016 | Triple Jump | ITA | 19
2016 | 100M | JAM | 11
2020 | 200M | JPN | 4
2016 | 100M | JPN | 12
2016 | 200M | JPN | 14
2016 | Long Jump | JPN | 17
2012 | 200M | JPN | 24
2008 | 200M | JPN | 34
2012 | Triple Jump | POR | 29
2008 | Tri

### Solución del requisito 2

Primero se obtiene el conjunto de campeones actuales para los dos eventos. Después se recupera el valor del año anterior dentro de cada historial de evento.

In [ ]:
sql_q2 = """
WITH champions AS (
    SELECT
        year,
        event,
        country AS current_champion
    FROM summer_medals
    WHERE gender = 'Men'
      AND medal = 'Gold'
      AND event IN ('Discus Throw', 'Triple Jump')
)
SELECT
    year,
    event,
    current_champion,
    LAG(current_champion, 1) OVER (
        PARTITION BY event
        ORDER BY year ASC
    ) AS previous_champion
FROM champions
ORDER BY event ASC, year ASC;
"""
q2_columns, q2_rows = run_sql(sql_q2)


year | event | current_champion | previous_champion
-------------------------------------------------------
2008 | Discus Throw | GER | NULL
2012 | Discus Throw | GER | GER
2016 | Discus Throw | ESP | GER
2020 | Discus Throw | ESP | ESP
2008 | Triple Jump | POR | NULL
2012 | Triple Jump | POR | POR
2016 | Triple Jump | ITA | POR
2020 | Triple Jump | ITA | ITA
Filas: 8


### Solución del requisito 3

La posición se reinicia para cada combinación de año y país y se define por el orden alfabético del evento.

In [ ]:
sql_q3 = """
SELECT
    year,
    country,
    event,
    ROW_NUMBER() OVER (
        PARTITION BY year, country
        ORDER BY event ASC
    ) AS country_year_position
FROM summer_medals
WHERE gender = 'Women'
  AND medal = 'Gold'
ORDER BY year ASC, country ASC, country_year_position ASC;
"""
q3_columns, q3_rows = run_sql(sql_q3)


year | country | event | country_year_position
--------------------------------------------------
2008 | CHN | 100M | 1
2008 | CHN | Long Jump | 2
2008 | ESP | Triple Jump | 1
2008 | FRA | Discus Throw | 1
2008 | JPN | 200M | 1
2012 | CHN | 100M | 1
2012 | CHN | Long Jump | 2
2012 | CHN | Triple Jump | 3
2012 | ESP | Discus Throw | 1
2012 | JPN | 200M | 1
2016 | CHN | Long Jump | 1
2016 | ESP | Discus Throw | 1
2016 | ESP | Triple Jump | 2
2016 | JPN | 100M | 1
2016 | JPN | 200M | 2
2020 | CHN | 100M | 1
2020 | CHN | Long Jump | 2
2020 | CHN | Triple Jump | 3
2020 | ESP | Discus Throw | 1
2020 | JPN | 200M | 1
Filas: 20


## Resultados obtenidos

Las salidas de las celdas anteriores proceden de la ejecución real de las consultas sobre los datos sintéticos creados en este notebook. Como comprobación adicional, se muestran algunas filas y el número total de filas de cada resultado.

In [ ]:
def show_sample(name, rows, n=8):
    print(f"{name}: {len(rows)} filas")
    for row in rows[:n]:
        print(row)
    if len(rows) > n:
        print("...")

show_sample("Requisito 1", q1_rows)
print()
show_sample("Requisito 2", q2_rows, n=10)
print()
show_sample("Requisito 3", q3_rows)


Requisito 1: 40 filas
(2020, '200M', 'CAN', 3)
(2020, '100M', 'CHN', 2)
(2020, 'Long Jump', 'CHN', 7)
(2020, 'Long Jump', 'CHN', 8)
(2020, 'Triple Jump', 'CHN', 10)
(2016, 'Long Jump', 'CHN', 18)
(2012, '100M', 'CHN', 22)
(2012, 'Long Jump', 'CHN', 28)
...

Requisito 2: 8 filas
(2008, 'Discus Throw', 'GER', None)
(2012, 'Discus Throw', 'GER', 'GER')
(2016, 'Discus Throw', 'ESP', 'GER')
(2020, 'Discus Throw', 'ESP', 'ESP')
(2008, 'Triple Jump', 'POR', None)
(2012, 'Triple Jump', 'POR', 'POR')
(2016, 'Triple Jump', 'ITA', 'POR')
(2020, 'Triple Jump', 'ITA', 'ITA')

Requisito 3: 20 filas
(2008, 'CHN', '100M', 1)
(2008, 'CHN', 'Long Jump', 2)
(2008, 'ESP', 'Triple Jump', 1)
(2008, 'FRA', 'Discus Throw', 1)
(2008, 'JPN', '200M', 1)
(2012, 'CHN', '100M', 1)
(2012, 'CHN', 'Long Jump', 2)
(2012, 'CHN', 'Triple Jump', 3)
...


## Interpretación y autoevaluación

La solución utiliza funciones de ventana porque necesita conservar cada fila y, al mismo tiempo, calcular información relativa a otras filas. En el primer resultado, una orden define las posiciones y otra orden distinta organiza la salida. En el segundo, la comparación histórica se mantiene dentro de cada evento, de modo que el primer año de cada evento no hereda el campeón de otro evento. En el tercero, la posición vuelve a empezar para cada combinación de año y país.

Para autoevaluarte, compara tus consultas con estas decisiones. Comprueba especialmente tres puntos: si el orden usado para calcular una posición coincide con el requisito, si una comparación histórica se mezcla accidentalmente entre eventos y si la numeración se reinicia exactamente en cada grupo de año y país. También revisa si tus resultados conservan todas las filas que debían aparecer y si los valores iniciales sin fila anterior quedan como `NULL`.